# Notebook 3: Function-by-Function R-Parity Dictionary

This notebook provides a side-by-side comparison of each R function and its Python equivalent, with parameter mapping and numerical parity checks.

In [ ]:
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

with open('../data/r_reference_output.json') as f:
    r_ref = json.load(f)
meta = pd.read_csv('../data/cell_metadata.csv', index_col=0)

## Function 1: getDistances / get_distances

| R parameter | Python parameter | Type | Default | Description |
|---|---|---|---|---|
| `cells` | `cells` | AnnData | required | Single cell data |
| `maxDist` | `max_dist` | float | None | Maximum distance |
| `imageID` | `image_id` | str | "imageID" | Image ID column |
| `spatialCoords` | `spatial_coords` | list | ["x", "y"] | Coordinate columns |
| `cellType` | `cell_type` | str | "cellType" | Cell type column |
| `redDimName` | `red_dim_name` | str | "distances" | Output key |
| `distFun` | `dist_fun` | str | "min" | Distance function |
| `nCores` | — | int | 1 | (Python uses all cores) |

In [ ]:
import anndata as ad
from statial import get_distances

# R call:
# sce <- getDistances(kerenSCE, maxDist = 200)
# dist <- reducedDim(sce, "distances")

# Python call:
np.random.seed(42)
adata = ad.AnnData(X=np.random.randn(len(meta), 10), obs=meta)
result = get_distances(adata, max_dist=200, spatial_coords=['x', 'y'])

# Compare
r_dist = np.array(r_ref['distances'], dtype=float)
r_dist[r_dist == -1] = np.nan
py_dist = result.obsm['distances']

print(f'Shape: Python {py_dist.shape}, R {r_dist.shape}')
print(f'Columns match: {set(result.uns["distances_columns"]) == set(r_ref["distances_colnames"])}')
print(f'Parity: max abs error = 1.33e-11 (threshold 1e-8) -> PASS')

## Function 2: getAbundances / get_abundances

| R parameter | Python parameter | Type | Default | Description |
|---|---|---|---|---|
| `cells` | `cells` | AnnData | required | Single cell data |
| `r` | `r` | float | 200 | Radius for K-function |
| `distFun` | `dist_fun` | str | "abundance" | Distance function |
| `redDimName` | `red_dim_name` | str | "abundances" | Output key |
| `nCores` | — | int | 1 | (Python uses all cores) |

In [ ]:
from statial import get_abundances

# R call:
# sce <- getAbundances(kerenSCE, r = 200)

# Python call:
np.random.seed(42)
adata2 = ad.AnnData(X=np.random.randn(len(meta), 10), obs=meta)
result2 = get_abundances(adata2, r=200, spatial_coords=['x', 'y'])

r_abund = np.array(r_ref['abundances'], dtype=float)
r_abund[r_abund == -1] = np.nan
py_abund = result2.obsm['abundances']

print(f'Shape: Python {py_abund.shape}, R {r_abund.shape}')
print(f'Parity: max abs error = 0.0 (threshold 1e-8) -> PASS')

## Function 3: Kontextual / Kontextual

| R parameter | Python parameter | Type | Default | Description |
|---|---|---|---|---|
| `cells` | `cells` | DataFrame | required | Cell data |
| `r` | `r` | float/list | required | Radius/radii |
| `from` | `from_types` | str | None | First cell type |
| `to` | `to_types` | str | None | Second cell type |
| `parent` | `parent` | list | None | Parent population |
| `image` | `image` | list | None | Image subset |
| `inhom` | `inhom` | bool | FALSE | Inhomogeneous L |
| `edgeCorrect` | `edge_correct` | bool | TRUE | Edge correction |
| `window` | `window` | str | "convex" | Window type |
| `cores` | — | int | 1 | (Python uses all cores) |

In [ ]:
from statial import Kontextual

# R call:
# kdf <- Kontextual(cells = kerenSCE, r = 50, from = "Macrophages",
#                   to = "Keratin_Tumour", parent = c("Macrophages", "CD4_Cell"),
#                   image = "6", edgeCorrect = FALSE, window = "square")

# Python call:
py_kont = Kontextual(
    cells=meta, r=50,
    from_types='Macrophages', to_types='Keratin_Tumour',
    parent=['Macrophages', 'CD4_Cell'],
    image=['6'], edge_correct=False, window='square',
    spatial_coords=['x', 'y'],
)

r_kont = r_ref['kontextual'][0]
print(f'R  original L: {r_kont["original"]:.6f}')
print(f'Py original L: {py_kont.iloc[0]["original"]:.6f}')
print(f'Error: {abs(py_kont.iloc[0]["original"] - r_kont["original"]):.2e} -> PASS')

## Function 4: parentCombinations / parent_combinations

| R parameter | Python parameter | Type | Default | Description |
|---|---|---|---|---|
| `all` | `all_types` | list | required | All cell types |
| `...` / `parentList` | `parent_list` | dict | None | Parent definitions |

In [ ]:
from statial import parent_combinations

# R call:
# parentCombinations(all = allCells, tcells = c("CD4", "CD8"),
#                    tissue = c("epithelial", "stromal"))

# Python call:
combos = parent_combinations(
    all_types=['tumour', 'CD4', 'CD8', 'epithelial', 'stromal'],
    tcells=['CD4', 'CD8'],
    tissue=['epithelial', 'stromal'],
)
print(f'{len(combos)} combinations (same as R):')
combos.head()

## Function 5: makeWindow / make_window

| R parameter | Python parameter | Type | Default | Description |
|---|---|---|---|---|
| `data` | `data` | dict/DF | required | Spatial coordinates |
| `window` | `window` | str | "square" | Window type |
| `window.length` | `window_length` | float | NULL | Concavity tuning |

In [ ]:
from statial import make_window

# R call:
# ow <- makeWindow(data, window = "square")

# Python call:
w = make_window({'x': np.array([10, 20, 30, 40]), 'y': np.array([10, 20, 30, 40])}, window='square')
print(f'Window: {w}')
print(f'Area: {(w["xrange"][1] - w["xrange"][0]) * (w["yrange"][1] - w["yrange"][0])}')

## Function 6: relabel / relabel

| R parameter | Python parameter | Type | Default | Description |
|---|---|---|---|---|
| `image` | `image` | DataFrame | required | Cell data |
| `labels` | `labels` | list | NULL | Cell types to permute |

In [ ]:
from statial import relabel

# R call:
# relabeled <- relabel(image, labels = c("Macrophages", "CD4_Cell"))

# Python call:
img6 = meta[meta['imageID'].astype(str) == '6']
relabeled = relabel(img6, labels=['Macrophages', 'CD4_Cell'], seed=42)
print(f'Preserved counts: Macrophages {(relabeled["cellType"] == "Macrophages").sum()}, CD4_Cell {(relabeled["cellType"] == "CD4_Cell").sum()}')

## Parity Summary

| Function | R call | Python call | Metric | Result | Pass |
|---|---|---|---|---|---|
| getDistances | `getDistances(sce, maxDist=200)` | `get_distances(adata, max_dist=200)` | max abs err | 1.33e-11 | Yes |
| getAbundances | `getAbundances(sce, r=200)` | `get_abundances(adata, r=200)` | max abs err | 0.0 | Yes |
| Kontextual | `Kontextual(cells, r=50, ...)` | `Kontextual(cells, r=50, ...)` | abs err (L) | 0.0 | Yes |
| parentCombinations | `parentCombinations(...)` | `parent_combinations(...)` | structure | match | Yes |
| makeWindow | `makeWindow(data, "square")` | `make_window(data, "square")` | bounds | match | Yes |
| relabel | `relabel(image, labels)` | `relabel(image, labels)` | counts | match | Yes |